In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
from model_utils.plots import plot_results
from lstm import LSTM
from dataset import TimeSeriesDataset
import time
import itertools
import math


In [2]:
import os
import shutil
import stat
import time

def handle_remove_readonly(func, path, exc):
    # Callback to handle read-only files on Windows
    excvalue = exc[1]
    if func in (os.rmdir, os.remove, os.unlink) and excvalue.errno == 13: # EACCES
        os.chmod(path, stat.S_IWRITE)
        func(path)
    else:
        raise

# Clean up directories from previous runs
dirs_to_cleanup = ['best_models', 'grid_search_plots', 'training_logs', 'inference_logs']
for dir_path in dirs_to_cleanup:
    if os.path.exists(dir_path):
        # Retry a few times in case of transient locks
        for i in range(3):
            try:
                shutil.rmtree(dir_path, ignore_errors=False, onerror=handle_remove_readonly)
                print(f"Removed directory: {dir_path}")
                break
            except Exception as e:
                if i < 2:
                    time.sleep(1) # Wait a bit before retrying
                else:
                    print(f"Error removing {dir_path}: {e}")

Removed directory: best_models
Removed directory: grid_search_plots
Removed directory: training_logs


In [3]:
import holidays

# -----------------------------------------------------------------------------
# DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
DATA_PATH = '../dataset/data_andre.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)


# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(["store_id", "item_id", DATE_COL]).reset_index(drop=True)

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]
print(len(y))

# Calandar-based features
# Ensure datetime
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

# Base calendar parts
df["day_of_week"]  = df[DATE_COL].dt.dayofweek.astype(int)
df["day_of_month"] = df[DATE_COL].dt.day.astype(int)         
df["moy"]          = df[DATE_COL].dt.month.astype(int)-1      
df["doy"]          = df[DATE_COL].dt.dayofyear.astype(int)-1   
df["is_weekend"] = (
    (df[DATE_COL].dt.dayofweek == 5) |
    (df[DATE_COL].dt.dayofweek == 6)
).astype(int)
# Create holiday calendar
df["time_idx"] = np.arange(len(df))
us_holidays = holidays.US()

# Holiday flag
df["is_holiday"] = df[DATE_COL].apply(lambda x: 1 if x in us_holidays else 0)

# Specific holidays
df["is_christmas"] = (df[DATE_COL].dt.month == 12) & (df[DATE_COL].dt.day == 25)

df["is_thanksgiving"] = df[DATE_COL].apply(
    lambda x: 1 if us_holidays.get(x) == "Thanksgiving Day" else 0
)

df["is_christmas"] = df["is_christmas"].astype(int)

# -----------------------------------------------------------------------------
# Add lag / rolling features
# -----------------------------------------------------------------------------
df["lag_1"] = df.groupby(["store_id", "item_id"])[TARGET_COL].shift(1)
df["lag_2"] = df.groupby(["store_id", "item_id"])[TARGET_COL].shift(2)
df["lag_3"] = df.groupby(["store_id", "item_id"])[TARGET_COL].shift(3)
df["lag_7"] = df.groupby(["store_id", "item_id"])[TARGET_COL].shift(7)
df["lag_14"] = df.groupby(["store_id", "item_id"])[TARGET_COL].shift(14)
df["lag_28"] = df.groupby(["store_id", "item_id"])[TARGET_COL].shift(28)

df["rolling_mean_3"] = df.groupby(["store_id", "item_id"])[TARGET_COL].transform(lambda x: x.shift(1).rolling(window=3).mean())
df["rolling_mean_7"] = df.groupby(["store_id", "item_id"])[TARGET_COL].transform(lambda x: x.shift(1).rolling(window=7).mean())

#LAG_COLS = ["lag_1", "lag_2", "lag_3", "lag_7"]
#ROLLING_COLS = ["rolling_mean_3", "rolling_mean_7"]
PROMO_BIN_COLS = [col for col in df.columns if col.startswith("promo_type_")]
PROMO_CONT_COLS = [col for col in df.columns if col.startswith("promo_value_")]
#df[LAG_COLS] = df[LAG_COLS].fillna(0)
#df[ROLLING_COLS] = df[ROLLING_COLS].fillna(0)

# Fill NaNs with 0 to keep the initial data rows instead of dropping them
df = df.fillna(0).reset_index(drop=True)
LAG_COLS = ["lag_1", "lag_2", "lag_3", "lag_7"]
#ROLLING_COLS = ["rolling_mean_3", "rolling_mean_7"]
CATEGORICAL_COLS = ["day_of_week", "doy", "moy", "day_of_month"]
#CURSE_OF_DIMENSIONALITY_COLS = ["doy"]  # This column creates 365 OHE columns, which is problematic for LSTM input
CONTINUOUS_COLS = LAG_COLS+ PROMO_CONT_COLS
BINARY_COLS = ["is_thanksgiving", "is_christmas", "is_weekend"] + PROMO_BIN_COLS

#EXOG_COLS = CATEGORICAL_COLS + CONTINUOUS_COLS + BINARY_COLS
EXOG_COLS = CONTINUOUS_COLS + BINARY_COLS # Exclude categorical for now to avoid OHE explosion
# Reorder columns to have date , target and exogenous variables first (optional, for better readability)
df = df[[DATE_COL, TARGET_COL] + [col for col in df.columns if col not in [DATE_COL, TARGET_COL] + EXOG_COLS] + EXOG_COLS]

print(PROMO_BIN_COLS)
print(PROMO_CONT_COLS)
# -----------------------------------------------------------------------------
# DATA SPLITTING
# -----------------------------------------------------------------------------
# Define split sizes
train_size = 455
val_size = 154
forecast_horizon = 152

# ensure day 2022-09-24 is the first day of test set
df = df.sort_values(["store_id", "item_id", DATE_COL]).reset_index(drop=True)

lookback_window = 7

Loading data from ../dataset/data_andre.feather...
1082371
['promo_type_FRPG', 'promo_type_GAS', 'promo_type_BOGO', 'promo_type_DISC', 'promo_type_CIRC', 'promo_type_CIRE', 'promo_type_CLCP', 'promo_type_LFPE']
['promo_value_FRPG', 'promo_value_GAS', 'promo_value_BOGO', 'promo_value_DISC', 'promo_value_CIRC', 'promo_value_CIRE', 'promo_value_CLCP', 'promo_value_LFPE']


In [4]:
df

,date,value,item_id,cat_label,sdep_label,dep_label,dmn_label,store_id,day_of_week,day_of_month,...,is_christmas,is_weekend,promo_type_FRPG,promo_type_GAS,promo_type_BOGO,promo_type_DISC,promo_type_CIRC,promo_type_CIRE,promo_type_CLCP,promo_type_LFPE
0,2021-01-23,8,27,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,6269,5,23,...,0,1,0,0,0,0,0,0,0,0
1,2021-01-24,14,27,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,6269,6,24,...,0,1,0,0,0,0,0,0,0,0
2,2021-01-25,8,27,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,6269,0,25,...,0,0,0,0,0,0,0,0,0,0
3,2021-01-26,5,27,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,6269,1,26,...,0,0,0,0,0,0,0,0,0,0
4,2021-01-27,8,27,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,6269,2,27,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1082366,2023-02-18,28,900087600,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,6269,5,18,...,0,1,0,0,0,0,0,0,0,0
1082367,2023-02-19,46,900087600,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,6269,6,19,...,0,1,0,0,0,0,0,0,0,0
1082368,2023-02-20,28,900087600,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,6269,0,20,...,0,0,0,0,0,0,0,0,0,0
1082369,2023-02-21,31,900087600,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,6269,1,21,...,0,0,0,0,0,0,0,0,0,0


In [5]:
print(df[TARGET_COL].describe())

count    1.082371e+06
mean     6.387899e+00
std      1.132080e+01
min      0.000000e+00
25%      1.000000e+00
50%      4.000000e+00
75%      8.000000e+00
max      1.000000e+03
Name: value, dtype: float64


# Grid Search Cell

In [6]:
import time
import itertools
import sys
import os
import importlib
sys.path.append(os.path.abspath('..'))

# Reload plot_results to pick up changes in model_utils.utils
import utils
importlib.reload(utils)
from utils import build_dynamic_exog_row, scale_exogenous_features, transform_exog_row
import model_utils.plots
importlib.reload(model_utils.plots)


# Modify lstm_experiment to return timings and handle plotting internally
def lstm_experiment_grid(df, target, item_id, store_id, train_size=500, val_size=100, forecast_window=161, 
                   seq_length=30, epochs=100, batch_size=32, lr=0.001, dropout=0.0, hidden_size=32, num_layers=1, 
                   categorical_cols=None, continuous_cols=None, binary_cols=None,
                   patience=50, seed=42, loss_type='MSELoss', save_plot_path=None, use_best_model=True):

    # Set seed for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    
    test_start_idx = len(df) - forecast_window
    val_start_idx = test_start_idx - val_size
    train_start_idx = val_start_idx - train_size

    print(f"Calculated indices - Train Start: {train_start_idx}, Val Start: {val_start_idx}, Test Start: {test_start_idx}")
    # Safety check
    if train_start_idx < 0:
        # If dataset is too short, we just start from 0
        train_start_idx = 0
        print(f"Warning: Dataset shorter than requested split sizes. adjusting train_start to 0.")

    train_slice = slice(train_start_idx, val_start_idx)
    val_slice = slice(val_start_idx, test_start_idx)
    test_slice = slice(test_start_idx, None)
    
    # Extract Target
    train = df[target][train_slice].values
    val = df[target][val_slice].values
    test = df[target][test_slice].values
    
    # Scale Target
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train.reshape(-1, 1)).flatten()
    val_scaled = scaler.transform(val.reshape(-1, 1)).flatten()

    # Extract Exogenous Variables
    exog_train_scaled = None
    exog_val_scaled = None
    exog_test_scaled = None
    
    # Keep track of combined exog correctly logically
    exog_cols = (categorical_cols or []) + (continuous_cols or []) + (binary_cols or [])
    if not exog_cols:
        exog_cols = None
        
    cat_scaler = None
    cont_scaler = None
    
    if exog_cols:
        # Get fully scaled dataframe and updated column names
        df_scaled, final_exog_cols, cat_scaler, cont_scaler = scale_exogenous_features(
            df, train_slice, categorical_cols, continuous_cols, binary_cols
        )
        
        # Extract slices directly from the new DataFrame
        exog_train_scaled = df_scaled[final_exog_cols].iloc[train_slice].values
        exog_val_scaled = df_scaled[final_exog_cols].iloc[val_slice].values
        exog_test_scaled = df_scaled[final_exog_cols].iloc[test_slice].values
        
    # Input size = 1 (target) + number of exog features
    input_size = 1 + (exog_train_scaled.shape[1] if exog_train_scaled is not None else 0)
    
    # -------------------------------------------------------------------------
    # 2. Datasets & Loaders
    # -------------------------------------------------------------------------
    # Pass exogenous data to TimeSeriesDataset
    train_dataset = TimeSeriesDataset(train_scaled, exog_train_scaled, seq_length)
    use_pin_memory = torch.cuda.is_available()
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, pin_memory=use_pin_memory)
    
    # Important: Pass exog_val to validation dataset too!
    val_dataset = TimeSeriesDataset(val_scaled, exog_val_scaled, seq_length)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, pin_memory=use_pin_memory)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout).to(device)
    
    if loss_type == 'MSELoss':
        criterion = nn.MSELoss()
    elif loss_type == 'L1Loss':
        criterion = nn.L1Loss()
    elif loss_type == 'HuberLoss':
        criterion = nn.HuberLoss()
    else:
        raise ValueError(f"Unsupported loss_type: {loss_type}")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion2 = nn.MSELoss()  # For validation loss calculation
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    
    # Directories
    model_dir = f'best_models/seed_{seed}/{loss_type}'
    os.makedirs(model_dir, exist_ok=True)
    best_model_path = f'{model_dir}/lstm_item{item_id}_store{store_id}.pth'
    
    log_dir = f'training_logs/seed_{seed}/{loss_type}'
    os.makedirs(log_dir, exist_ok=True)
    log_path = f'{log_dir}/lstm_item{item_id}_store{store_id}_log.csv'

    best_epoch = 0
    
    # -------------------------------------------------------------------------
    # 3. Training Loop
    # -------------------------------------------------------------------------
    start_train_time = time.time()
    
    with open(log_path, 'w') as log_file:
        log_file.write("Epoch,Batch,Batch_X,Batch_Y,Output,Loss\n")
        
        for epoch in range(epochs):
            model.train()
            epoch_loss = 0
            for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                optimizer.zero_grad()
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                
                # Convert tensors to lists (represented as strings) to log in CSV
                batch_x_str = str(batch_x.cpu().detach().numpy().tolist()).replace('"', "'")
                batch_y_str = str(batch_y.cpu().detach().numpy().tolist()).replace('"', "'")
                outputs_str = str(outputs.cpu().detach().numpy().tolist()).replace('"', "'")
                
                log_file.write(f'{epoch},{batch_idx},"{batch_x_str}","{batch_y_str}","{outputs_str}",{loss.item()}\n')
            
            avg_train_loss = epoch_loss / len(train_loader)
            train_losses.append(avg_train_loss)
            
            # Validation
            model.eval()
            all_outputs = []
            all_targets = []

            with torch.no_grad():
                for batch_idx, (batch_x, batch_y) in enumerate(val_loader):
                    batch_x = batch_x.to(device)
                    batch_y = batch_y.to(device)
                    
                    outputs = model(batch_x)
                    all_outputs.append(outputs)
                    all_targets.append(batch_y)

            all_outputs = torch.cat(all_outputs, dim=0).view(-1)
            all_targets = torch.cat(all_targets, dim=0).view(-1)
            
            val_loss = criterion2(all_outputs, all_targets).item()
            val_losses.append(val_loss)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch + 1
                if use_best_model:
                    torch.save(model.state_dict(), best_model_path)
                patience_counter = 0  # Reset patience counter


            patience_counter += 1
            if patience_counter >= patience:  
                print(f"Early stopping at epoch {epoch+1} due to no improvement in validation loss for {patience} epochs.")
                break  
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {val_loss:.6f}")

    if not use_best_model:
        torch.save(model.state_dict(), best_model_path)
        best_epoch = epochs
        print(f"Saved last model (Epoch {epochs}) to {best_model_path}")
    else:
        print(f"Saved best model (Epoch {best_epoch}) to {best_model_path}")

    train_time = time.time() - start_train_time

    # -------------------------------------------------------------------------
    # 4. Inference (Recursive)
    # -------------------------------------------------------------------------
    start_inference_time = time.time()
    
    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path))
        if use_best_model:
            print (f"Loaded model from: {best_model_path} (Best Epoch: {best_epoch})")
        else:
            print (f"Loaded model from: {best_model_path} (Last Epoch)")
    else:
        print("Warning: No model saved. Using current model in memory.")

    model.eval()
    
    # Get the start date of the test set
    test_start_date = df[DATE_COL].iloc[test_start_idx]
    print(f"Test set starts on: {test_start_date.date()}")

    # Use the last seq_length points from validation data as initial input
    current_seq = val_scaled[-seq_length:].tolist()
    
    # Extended unscaled history specifically for lag/rolling exogenous features
    if exog_cols is not None:
        # Shift exog by 1 to match training dataloader
        current_exog_seq = exog_val_scaled[-seq_length + 1:].tolist() + [exog_test_scaled[0].tolist()]
        
        # Calculate max history needed for dynamically calculating lag/rolling features
        lag_cols = [c for c in exog_cols if c.startswith('lag_')]
        rolling_cols = [c for c in exog_cols if c.startswith('rolling_')]
        max_lookback = seq_length
        if lag_cols:
            max_lookback = max(max_lookback, max([int(c.split('_')[1]) for c in lag_cols]))
        if rolling_cols:
            max_lookback = max(max_lookback, max([int(c.split('_')[2]) for c in rolling_cols]))
            
        full_scaled = np.concatenate([train_scaled, val_scaled])
        full_unscaled = scaler.inverse_transform(full_scaled.reshape(-1, 1)).flatten().tolist()
        current_raw_hist = full_unscaled[-max_lookback:]
        
    current_date_seq = df[DATE_COL].iloc[test_start_idx - seq_length : test_start_idx].dt.date.tolist()
    
    forecast = []
    
    if exog_cols is not None:
        # Teacher-forced diagnostic: rebuilt rows should match df_scaled rows
        hist = full_unscaled[-max_lookback:].copy()

        for k in range(min(3, forecast_window - 1)):
            # append TRUE raw test value, not prediction
            hist.append(test[k])
            hist = hist[-max_lookback:]

            idx = test_start_idx + k + 1
            next_date = df[DATE_COL].iloc[idx]
            df_row = df.iloc[idx]

            rebuilt_raw = build_dynamic_exog_row(
                next_date=next_date,
                target_history=hist,
                exog_cols=exog_cols,
                lag_cols=lag_cols,
                rolling_cols=rolling_cols,
                df_row=df_row
            )

            rebuilt_scaled = transform_exog_row(
                rebuilt_raw,
                categorical_cols=categorical_cols,
                continuous_cols=continuous_cols,
                binary_cols=binary_cols,
                cat_scaler=cat_scaler,
                cont_scaler=cont_scaler,
                final_exog_cols=final_exog_cols
            )

            ref = df_scaled[final_exog_cols].iloc[[idx]]

            diff = np.max(np.abs(rebuilt_scaled.values - ref.values))
            print(f"k={k}, idx={idx}, max_abs_diff={diff}")
    # Setup inference log file
    inf_log_dir = f'inference_logs/seed_{seed}/{loss_type}'
    os.makedirs(inf_log_dir, exist_ok=True)
    inf_log_path = f'{inf_log_dir}/inference_item{item_id}_store{store_id}.csv'
    
    with open(inf_log_path, 'w') as inf_log_file:
        inf_log_file.write("Step,X,Predicted_Y\n")
        
        with torch.no_grad():
            for step in range(forecast_window):
                # Prepare input
                if exog_cols is not None:
                    # Combine target sequence with exog
                    current_seq_arr = np.array(current_seq).reshape(-1, 1)
                    current_exog_arr = np.array(current_exog_seq)
                    x_np = np.column_stack([current_seq_arr, current_exog_arr])
                else:
                    x_np = np.array(current_seq).reshape(-1, 1)
                
                x = torch.FloatTensor(x_np).unsqueeze(0).to(device)
                
                # Predict next value
                pred = model(x).cpu().numpy()[0, 0]
                forecast.append(pred)
                
                # Format X to log
                x_str = str(x_np.tolist()).replace('"', "'")
                inf_log_file.write(f'{step},"{x_str}",{pred}\n')
                
                # Print date info
                # Print date info
                y_date = df[DATE_COL].iloc[test_start_idx + step].date()
                print(f"Step {step}: Predicting for Date: {y_date}")
                print(f"X element dates (len {len(current_date_seq)}): {current_date_seq}")
                
                # Update sequence (sliding window)
                # pred is model output in scaled target space
                current_seq = current_seq[1:] + [pred]
                current_date_seq = current_date_seq[1:] + [y_date]

                if exog_cols is not None:
                    # Convert prediction back to RAW target space
                    pred_raw = scaler.inverse_transform(np.array(pred).reshape(-1, 1)).flatten()[0]

                    # Update raw history
                    current_raw_hist.append(pred_raw)
                    current_raw_hist = current_raw_hist[-max_lookback:]

                    # Update exog sequence for the NEXT step
                    if step + 1 < forecast_window:
                        # Target index for the next step's future exogenous data
                        target_idx = test_start_idx + step + 1
                        next_date = df[DATE_COL].iloc[target_idx]
                        if isinstance(next_date, pd.Timestamp):
                            next_date = next_date.date()

                        # Get known RAW exogenous data for this next date (e.g., promos, events)
                        df_row = df.iloc[target_idx]
                        print(f"Step {step}: Next date {next_date}, RAW exog row: {df_row.to_dict()}")
                        # Build RAW exog row
                        next_exog_row_raw = build_dynamic_exog_row(
                            next_date=next_date,
                            target_history=current_raw_hist,
                            exog_cols=exog_cols,
                            lag_cols=lag_cols,
                            rolling_cols=rolling_cols,
                            df_row=df_row
                        )

                        # Transform to MODEL scaled space
                        next_exog_row_scaled = transform_exog_row(
                            next_exog_row_raw,
                            categorical_cols=categorical_cols,
                            continuous_cols=continuous_cols,
                            binary_cols=binary_cols,
                            cat_scaler=cat_scaler,
                            cont_scaler=cont_scaler,
                            final_exog_cols=final_exog_cols
                        )

                        # Append scaled row to exog sequence
                        current_exog_seq = current_exog_seq[1:] + [next_exog_row_scaled.iloc[0].tolist()]
                
                
    # Inverse transform predictions
    forecast = scaler.inverse_transform(np.array(forecast).reshape(-1, 1)).flatten()
    print(f"Forecasted values {forecast[:]}")
    inference_time = time.time() - start_inference_time
    
    # Metrics
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mae = mean_absolute_error(test, forecast)
    bias = np.mean(forecast - test)
    score = 0.5 * rmse + 0.25 * mae + 0.25 * abs(bias) 
    if save_plot_path:
        train_index = df[DATE_COL][train_slice].values
        val_index = df[DATE_COL][val_slice].values
        test_index = df[DATE_COL][test_slice].values
        
        plot_results(train, val, test, forecast, train_index, val_index, test_index, 
                     train_losses, val_losses, target_col=target, 
                     title=f'LSTM Forecast (Seed={seed}, Loss={loss_type}, Item={item_id}, Store={store_id})',
                     save_path=save_plot_path,
                     rmse=rmse, mae=mae, bias=bias, score=score, df_full=df)
    
    return rmse, mae, train_time, inference_time, best_epoch

In [7]:
# Grid Search Parameters
seeds = [2024]
loss_functions = ['MSELoss']  
batch_size = 32
hidden_size = 32
dropout = 0.0
EPOCHS = 1000 # Ensure EPOCHS is defined
LEARNING_RATE = 0.001

# Filter for specific products if needed
target_products = [916110]
if target_products:
    products = df[df['item_id'].isin(target_products)][['item_id', 'store_id']].drop_duplicates().values
else:
    # Get all unique products from the subset dataset
    products = df[['item_id', 'store_id']].drop_duplicates().values

# Prepare results storage
results = []
os.makedirs('grid_search_plots', exist_ok=True)


In [8]:
# check products series count and check for any missing dates
item_id, store_id = products[0]  # Just checking the first product for now
# Filter data for the specific product
df_product = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
df_product

,date,value,item_id,cat_label,sdep_label,dep_label,dmn_label,store_id,day_of_week,day_of_month,...,is_christmas,is_weekend,promo_type_FRPG,promo_type_GAS,promo_type_BOGO,promo_type_DISC,promo_type_CIRC,promo_type_CIRE,promo_type_CLCP,promo_type_LFPE
1028440,2021-01-23,60,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,6269,5,23,...,0,1,0,0,0,0,0,0,0,0
1028441,2021-01-24,29,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,6269,6,24,...,0,1,0,0,0,0,0,0,0,0
1028442,2021-01-25,5,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,6269,0,25,...,0,0,0,0,0,0,0,0,0,0
1028443,2021-01-26,13,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,6269,1,26,...,0,0,0,0,0,0,0,0,0,0
1028444,2021-01-27,31,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,6269,2,27,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1029196,2023-02-18,41,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,6269,5,18,...,0,1,0,0,0,0,0,0,0,0
1029197,2023-02-19,35,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,6269,6,19,...,0,1,0,0,0,0,0,0,0,0
1029198,2023-02-20,23,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,6269,0,20,...,0,0,0,0,0,0,0,0,0,0
1029199,2023-02-21,25,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,6269,1,21,...,0,0,0,0,0,0,0,0,0,0


In [ ]:


# Prepare results storage
results = []
os.makedirs('grid_search_plots', exist_ok=True)

# Run Grid Search
print(f"Starting Grid Search with {len(seeds)} seeds, {len(loss_functions)} loss functions and {len(products)} products...")

for seed in seeds:
    print(f"\n--- Processing Seed: {seed} ---")
    for loss_type in loss_functions:
        print(f"\n--- Processing Loss Type: {loss_type} ---")
        
        for item_id, store_id in products:
            print(f"Running: Seed={seed}, Loss={loss_type}, Item={item_id}, Store={store_id}")
            
            # Filter data for the specific product
            df_product = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
            
            # Handle DATE_COL (ensure it is a column and not in the index)
            if DATE_COL in df_product.index.names:
                if DATE_COL in df_product.columns:
                    # If it's in both, drop the index version to avoid "cannot insert" error
                    df_product = df_product.reset_index(drop=True)
                else:
                    # If it's only in the index, move it to a column
                    df_product = df_product.reset_index()

            # Fallback: simple reset to ensure RangeIndex 0..N
            df_product = df_product.reset_index(drop=True)

            df_product[DATE_COL] = pd.to_datetime(df_product[DATE_COL])
            df_product = df_product.sort_values(DATE_COL)
            df_product = df_product.reset_index(drop=True) # Final clean reset
            
            # Create directory for plots if it doesn't exist
            plot_dir = f'grid_search_plots/seed_{seed}/{loss_type}'
            os.makedirs(plot_dir, exist_ok=True)
            plot_filename = f'{plot_dir}/lstm_item{item_id}_store{store_id}.png'
            rmse, mae, train_time, infer_time, best_epoch = lstm_experiment_grid(
                df=df_product, 
                target=TARGET_COL, 
                item_id=item_id,
                store_id=store_id,
                train_size=train_size, 
                val_size=val_size,
                forecast_window=forecast_horizon, 
                seq_length=lookback_window,
                epochs=EPOCHS,  # Use the global EPOCHS setting (e.g., 1000)
                batch_size=batch_size, 
                lr=LEARNING_RATE,
                dropout=dropout,
                hidden_size=hidden_size,
                num_layers=1,
                patience=100,
                categorical_cols=CATEGORICAL_COLS, # Exclucluded for now to avoid OHE explosion as intended
                continuous_cols=CONTINUOUS_COLS,
                binary_cols=BINARY_COLS,
                seed=seed,
                loss_type=loss_type,
                save_plot_path=plot_filename
            )
            
            results.append({
                'seed': seed,
                'loss_type': loss_type,
                'item_id': item_id,
                'store_id': store_id,
                'batch_size': batch_size,
                'hidden_size': hidden_size,
                'dropout': dropout,
                'rmse': rmse,
                'mae': mae,
                'train_time': train_time,
                'inference_time': infer_time,
                'best_epoch': best_epoch,
                'plot_path': plot_filename
            })
        

# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df.to_csv('grid_search_results.csv', index=False)

Starting Grid Search with 1 seeds, 1 loss functions and 1 products...

--- Processing Seed: 2024 ---

--- Processing Loss Type: MSELoss ---
Running: Seed=2024, Loss=MSELoss, Item=916110, Store=6269
Calculated indices - Train Start: 0, Val Start: 455, Test Start: 609
Epoch 10/1000 | Train Loss: 0.021799 | Val Loss: 0.021595
Epoch 20/1000 | Train Loss: 0.018893 | Val Loss: 0.018667
Epoch 30/1000 | Train Loss: 0.018172 | Val Loss: 0.018307
Epoch 40/1000 | Train Loss: 0.017795 | Val Loss: 0.018223
Epoch 50/1000 | Train Loss: 0.017507 | Val Loss: 0.018245
Epoch 60/1000 | Train Loss: 0.017258 | Val Loss: 0.018307
Epoch 70/1000 | Train Loss: 0.017032 | Val Loss: 0.018379
Epoch 80/1000 | Train Loss: 0.016822 | Val Loss: 0.018450
Epoch 90/1000 | Train Loss: 0.016627 | Val Loss: 0.018515
Epoch 100/1000 | Train Loss: 0.016443 | Val Loss: 0.018581
Epoch 110/1000 | Train Loss: 0.016264 | Val Loss: 0.018661
Epoch 120/1000 | Train Loss: 0.016082 | Val Loss: 0.018763
Epoch 130/1000 | Train Loss: 0.015